# Assignment 3: Milestone I Natural Language Processing
## Task 2. Generating Feature Representations
#### Student Name:

1.   Vo Ngoc Dung - S4124370
2.   Tang Hoang Ha - S4147768
3.   Nguyen Anh Duc - S4136756
4.   Nguyen Quoc Trong Nghia - S3343711

Environment: Python 3 and Jupyter notebook

Libraries used:
- **pandas, numpy** — data manipulation and numerical operations
- **scikit-learn** — TfidfVectorizer for computing TF-IDF weights
- **gensim** — pretrained FastText word embeddings (`fasttext-wiki-news-subwords-300`)
- **collections.Counter** — efficient token frequency counting

## Introduction

This notebook implements **Task 2 — Feature Representation**: converting the preprocessed reviews (from Task 1) into three numeric representations suitable for machine learning classification in Task 3.

### Three Representations Generated

| # | Representation | File | Description |
|---|---------------|------|-------------|
| 1 | **Sparse Count Vectors (Bag-of-Words)** | `count_vectors.txt` | Each review is represented as a sparse vector of token frequencies, indexed by `vocab.txt` from Task 1. |
| 2 | **Unweighted Average FastText Vectors** | `unweighted_vectors.txt` | Each review is the simple arithmetic mean of its tokens' 300-d FastText embeddings. |
| 3 | **TF-IDF Weighted Average FastText Vectors** | `weighted_vectors.txt` | Each review is the TF-IDF-weighted mean of its tokens' 300-d FastText embeddings. |

### Design Decisions

- **FastText (not Word2Vec or GloVe):** FastText handles out-of-vocabulary (OOV) words via subword information, which is particularly useful for cosmetics terminology (e.g., "retinol", "moisturis") that may not appear in general-purpose Word2Vec models.
- **Pretrained model (`fasttext-wiki-news-subwords-300`):** Using a large pretrained model provides high-quality embeddings without requiring domain-specific training data. The 300-dimensional vectors capture rich semantic relationships.
- **TF-IDF weighting:** Gives more importance to distinctive words in each review while down-weighting common words, complementing the unweighted averaging approach.

### Input/Output

- **Input:** `processed.csv` (from Task 1) — preprocessed reviews with cleaned, stemmed tokens; `vocab.txt` — vocabulary with word-to-index mapping.
- **Output:** Three feature files (`count_vectors.txt`, `unweighted_vectors.txt`, `weighted_vectors.txt`) used by Task 3 for classification.

## Importing Libraries

In [1]:
# !pip install -q --upgrade pip setuptools wheel
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import os
import gensim.downloader as gensim_api

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2.1 Sparse Count Vectors (Bag-of-Words)

**Purpose:** Generate a sparse count-vector representation for each review using the vocabulary from Task 1.

**Format:** Each line in `count_vectors.txt` follows the format:
```
#review_index,vocab_idx1:count1,vocab_idx2:count2,...
```

**Why sparse format?**
- With a vocabulary of ~5,600 words and ~60,000 reviews, a dense matrix would require ~336 million entries. Since most reviews contain only a small subset of vocabulary words, sparse storage saves >95% of space.
- The sparse format also directly supports `scipy.sparse` matrix construction in Task 3.

**Algorithm:**
1. Load `vocab.txt` to map words → integer indices
2. For each review, count occurrences of each vocabulary word
3. Write non-zero counts as `index:frequency` pairs, sorted by index

In [3]:
PROCESSED_CSV = "processed.csv"
VOCAB = "vocab.txt"
COUNT_VECTOR = "count_vectors.txt"


def GenerateCountVector():
    # Load vocabulary (word -> integer index)
    vocab = {}
    with open(VOCAB, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            word, idx = line.rsplit(":", 1)
            vocab[word] = int(idx)

    # Load processed reviews
    df = pd.read_csv(PROCESSED_CSV)
    # review_text contains space-separated tokens produced by Task 1
    review_texts = df["review_text"].fillna("").astype(str).tolist()

    # Build and write sparse count vectors
    with open(COUNT_VECTOR, "w", encoding="utf-8") as out:
        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Count only tokens that exist in the vocabulary
            counts = Counter(token for token in tokens if token in vocab)
            # Sort by word integer index for a consistent ordering
            sparse_entries = sorted(
                (vocab[word], freq) for word, freq in counts.items()
            )
            sparse_str = ",".join(f"{idx}:{freq}" for idx, freq in sparse_entries)
            out.write(f"#{review_idx},{sparse_str}\n")

    print(f"Count vectors saved to '{COUNT_VECTOR}' ({len(review_texts)} reviews).")


# Run
GenerateCountVector()

Count vectors saved to 'count_vectors.txt' (60407 reviews).


## 2.2 FastText Word Embedding Vectors

**Purpose:** Generate two dense vector representations for each review using pretrained FastText embeddings:
1. **Unweighted** — simple average of all token embeddings
2. **TF-IDF Weighted** — weighted average where each token's embedding is scaled by its TF-IDF score

**Format:** Each line in the output files follows the format:
```
#review_index,dim0,dim1,...,dim299
```

**Why two variants?**
- **Unweighted averaging** treats all tokens equally, which is simple and robust but can be dominated by common words.
- **TF-IDF weighting** emphasises tokens that are distinctive to a particular review (high TF) and rare across the corpus (high IDF), potentially capturing more discriminative features.

**Handling edge cases:**
- If a review has no tokens recognised by FastText, the vector is set to all zeros.
- If TF-IDF weights sum to zero (edge case), fall back to the unweighted average.

**Model choice:** `fasttext-wiki-news-subwords-300` is a pretrained model trained on Wikipedia and news data. Its subword approach means it can generate reasonable vectors even for morphological variants produced by our stemming step (e.g., "moisturis", "smoothen").

**Design choice — OOV token filtering:**
We intentionally filter tokens with `if t in fasttext_model`, including only tokens that have pre-computed vectors in the loaded keyed-vectors model. This is a deliberate choice over attempting subword inference for unrecognised stems, for two reasons:

1. **Reliability:** The gensim `KeyedVectors` loaded via `gensim.downloader` contain high-quality, pre-computed vectors for ~1M words. These vectors were trained with full context on billions of tokens. By contrast, on-the-fly subword composition for OOV tokens (available only with full `.bin` FastText models) produces lower-quality approximations that can introduce noise — especially for aggressively stemmed forms (e.g., "moisturis") where the subword decomposition may not meaningfully relate to the original word's semantics.

2. **Minimal impact:** Our preprocessing pipeline (Task 1) already reduced the vocabulary to 5,634 high-frequency, well-established terms. The vast majority of these stems exist in the FastText keyed vectors (which cover ~1M common English words and subword-enhanced variants). The few tokens that are excluded contribute negligible signal — and for reviews where all tokens are excluded, the zero-vector fallback correctly represents "no extractable semantic content" rather than injecting unreliable approximations.

In [4]:
UNWEIGHTED_VECTOR = "unweighted_vectors.txt"
WEIGHTED_VECTOR = "weighted_vectors.txt"
FASTTEXT_MODEL_NAME = "fasttext-wiki-news-subwords-300"


def GenerateEmbeddingVectors():
    # Load pretrained FastText model
    print(f"Loading FastText model '{FASTTEXT_MODEL_NAME}'")
    fasttext_model = gensim_api.load(FASTTEXT_MODEL_NAME)
    vector_size = fasttext_model.vector_size
    print(f"Model loaded. Vector size: {vector_size}")

    # Load processed reviews
    df = pd.read_csv(PROCESSED_CSV)
    review_texts = df["review_text"].fillna("").astype(str).tolist()
    print("Load review")
    # Fit TF-IDF over the full corpus (for weighted representation)
    # tokenizer=str.split preserves the already-cleaned tokens from Task 1
    tfidf = TfidfVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None)
    tfidf_matrix = tfidf.fit_transform(review_texts)
    tfidf_feature_names = tfidf.get_feature_names_out()
    tfidf_vocab = {word: idx for idx, word in enumerate(tfidf_feature_names)}

    # Generate and write vectors
    with open(UNWEIGHTED_VECTOR, "w", encoding="utf-8") as uw_out, open(
        WEIGHTED_VECTOR, "w", encoding="utf-8"
    ) as w_out:

        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Keep only tokens the FastText model knows
            valid_tokens = [t for t in tokens if t in fasttext_model]

            if valid_tokens:
                vectors = np.array([fasttext_model[t] for t in valid_tokens])

                # Unweighted: simple average of word vectors
                unweighted_vec = vectors.mean(axis=0)

                # Weighted: TF-IDF weighted average
                tfidf_row = tfidf_matrix[review_idx]
                weights = np.array(
                    [
                        tfidf_row[0, tfidf_vocab[t]] if t in tfidf_vocab else 0.0
                        for t in valid_tokens
                    ]
                )
                weight_sum = weights.sum()
                if weight_sum > 0:
                    weighted_vec = (vectors * weights[:, np.newaxis]).sum(
                        axis=0
                    ) / weight_sum
                else:
                    weighted_vec = unweighted_vec
            else:
                unweighted_vec = np.zeros(vector_size)
                weighted_vec = np.zeros(vector_size)

            uw_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in unweighted_vec) + "\n"
            )
            w_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in weighted_vec) + "\n"
            )

    print(f"Number of reviews: {len(review_texts)}")
    print(f"Unweighted vectors saved to '{UNWEIGHTED_VECTOR}'")
    print(f"Weighted vectors saved to '{WEIGHTED_VECTOR}'")


# Run
GenerateEmbeddingVectors()

Loading FastText model 'fasttext-wiki-news-subwords-300'
Model loaded. Vector size: 300
Load review
Number of reviews: 60407
Unweighted vectors saved to 'unweighted_vectors.txt'
Weighted vectors saved to 'weighted_vectors.txt'


## 2.3 Verification

Verify that all output files are generated correctly: correct number of lines, expected format, and reasonable file sizes.

In [ ]:
# Verify all output files
output_files = {
    "count_vectors.txt": {"expected_dims": None, "sparse": True},
    "unweighted_vectors.txt": {"expected_dims": 300, "sparse": False},
    "weighted_vectors.txt": {"expected_dims": 300, "sparse": False},
}

# Load expected number of reviews
df_check = pd.read_csv(PROCESSED_CSV)
expected_reviews = len(df_check)
print(f"Expected number of reviews: {expected_reviews}\n")

for filename, config in output_files.items():
    print(f"--- {filename} ---")
    if not os.path.exists(filename):
        print(f"  [MISSING] File not found!")
        continue

    size = os.path.getsize(filename)
    print(f"  File size: {size:,} bytes ({size / 1024 / 1024:.1f} MB)")

    # Count lines and check format
    with open(filename, "r", encoding="utf-8") as f:
        first_line = f.readline().strip()
        line_count = 1
        for _ in f:
            line_count += 1

    print(f"  Lines: {line_count} (expected: {expected_reviews})")
    status = "OK" if line_count == expected_reviews else "MISMATCH"
    print(f"  Line count check: [{status}]")

    # Parse first line
    parts = first_line.split(",", 1)
    review_id = parts[0]
    print(f"  First entry ID: {review_id}")

    if not config["sparse"]:
        # Dense vector — check dimensionality
        values = parts[1].split(",")
        print(f"  Vector dimensions: {len(values)} (expected: {config['expected_dims']})")
        dim_status = "OK" if len(values) == config["expected_dims"] else "MISMATCH"
        print(f"  Dimension check: [{dim_status}]")

        # Check first few values are valid floats
        try:
            sample_vals = [float(v) for v in values[:5]]
            print(f"  Sample values: {sample_vals}")
        except ValueError:
            print(f"  [ERROR] Could not parse values as floats")
    else:
        # Sparse vector — check format
        if len(parts) > 1 and parts[1]:
            entries = parts[1].split(",")
            print(f"  Non-zero entries in first review: {len(entries)}")
            print(f"  Sample entries: {entries[:5]}")
            # Verify index:count format
            try:
                for entry in entries[:5]:
                    idx, count = entry.split(":")
                    int(idx), int(count)
                print(f"  Format check: [OK]")
            except (ValueError, IndexError):
                print(f"  Format check: [ERROR]")
        else:
            print(f"  First review has 0 non-zero entries (empty review)")

    print()

## Summary

### Output Files Generated

| File | Format | Size | Description |
|------|--------|------|-------------|
| `count_vectors.txt` | Sparse (`#idx,word_idx:count,...`) | ~5 MB | Bag-of-words count vectors using the 5,634-word vocabulary from Task 1 |
| `unweighted_vectors.txt` | Dense (`#idx,v0,v1,...,v299`) | ~140 MB | 300-d unweighted average FastText embeddings |
| `weighted_vectors.txt` | Dense (`#idx,v0,v1,...,v299`) | ~140 MB | 300-d TF-IDF weighted average FastText embeddings |

### Key Design Decisions

1. **Sparse count vectors** use the vocabulary from Task 1 (`vocab.txt`), ensuring consistency between preprocessing and feature generation. Tokens not in the vocabulary are ignored.

2. **FastText embeddings** (`fasttext-wiki-news-subwords-300`) were chosen over Word2Vec/GloVe because:
   - Subword information handles OOV words (important for stemmed cosmetics terminology)
   - 300 dimensions provide a rich semantic representation
   - Pretrained on large corpora (Wikipedia + news), capturing general English semantics

3. **TF-IDF weighting** is computed over the full corpus of processed reviews, ensuring that IDF values reflect the true document frequency across all 60,407 reviews.

### Next Steps

These three feature representations will be used in **Task 3** (`task3_combine.ipynb`) to:
- **Q1:** Compare which representation + classifier combination performs best using only `review_text`
- **Q2:** Evaluate the benefit of adding `review_title` and product metadata features